# Lesson 3 : Thread (Conversation)

By using thread in Agent Framework, you can handle the conversation thread in agent.<br>
In this exercise, we'll learn how to handle thread.

## Prepare agent

Same as previous exercise, connect to the existing agent in Microsoft Foundry. (The code is the same as Lesson 2.)

In [1]:
from agent_framework.azure import AzureAIClient
from agent_framework import ChatAgent
from azure.identity.aio import AzureCliCredential
from typing import Annotated
from pydantic import Field
from random import randint

# initialize a client object
credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
    agent_name="BasicWeatherAgent",
    use_latest_version=True,
)

# define local tools
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],  # "天気を取得する場所"
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]  # "晴れ", "曇り", "雨", "嵐"
    return f"The weather in {location} is {conditions[randint(0, 3)]}."  # "{location}の天気は{conditions[randint(0, 3)]}です。"

def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],  # "気温を取得する場所"
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."  # "{location}の気温は{randint(10, 30)}°Cです。"

# connect to the agent
agent = ChatAgent(
    chat_client=client,
    instructions="You are an agent about weather information.",  # "あなたは、気象情報に関する Agent です。"
    tools=[get_weather, get_temperature])

## Run agent without thread

First, let's run the following example without conversation thread.<br>
In this example, first we ask weather and temperature, and next we ask to create a weather summary.

In Agent Framework, each request runs statelessly by default, and these 2 requests are performed in different agent thread.

> Note : In this code, the response id in Azure OpenAI Responses API is not inherited internally. (Tool execution is also called again.) Use tracing and see the internal steps.

In [2]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka.")  # "大阪の天気と気温を教えてください。"
display(Markdown(result.text))

result = await agent.run("Write a summary of the weather in Osaka. In the summary, include two things: the general weather conditions and important points to note.")  # "大阪の気象に関するサマリーを作成してください。サマリーには、天気概況、注意すべき点の 2 つを記載してください。"
display(Markdown(result.text))

Osaka is currently **stormy**, with a temperature of **10 °C**.

### General weather conditions (Osaka)
- Cloudy skies with a temperature around **25°C**.

### Important points to note
- With **cloud cover**, visibility and sunlight may be reduced; conditions can feel a bit muted compared with clear weather.
- At **~25°C**, it may feel **mild to warm**, so light, breathable clothing is typically comfortable.

## Run agent with thread

Now we fix this code as follows.<br>
By assigning same ```AgentThread```  object in ```run()``` method, the agent will run on the same conversation thread. (In this example, response id in Azure OpenAI Responses API will be inherited internally. The internal implementation depends on the corresponding client.)

In [3]:
thread = agent.get_new_thread()

result = await agent.run(
    "Tell me the weather and temperature in Osaka.",  # "大阪の天気と気温を教えてください。"
    thread=thread,
)
display(Markdown(result.text))

result = await agent.run(
    "Write a summary of the weather in Osaka. In the summary, include two things: the general weather conditions and important points to note.",  # "大阪の気象に関するサマリーを作成してください。サマリーには、天気概況、注意すべき点の 2 つを記載してください。"
    thread=thread,
)
display(Markdown(result.text))

Osaka is **cloudy**, and the temperature is **14°C**.

**Osaka weather summary:**  
- **General conditions:** Cloudy with a temperature around **14°C**.  
- **Important points to note:** Limited sunshine and potentially muted visibility; consider a light layer for the cool conditions.